# R06 · Confidence intervals and the numbers quoted in the paper

Reviewer C / supervisor plan: windows from the same file are strongly dependent (stride 1), so CIs are
computed by **resampling whole files** (file-level bootstrap, 1000 resamples), not single windows.
Computed for seed 0 of each main model; the seed-to-seed spread is in R03.

In [1]:
# ── Settings ──────────────────────────────────────────────────────────────
import os
WORKERS = int(os.environ.get("REV_WORKERS", 8))   # training jobs side by side on the GPU
DEVICE  = None     # None = use the GPU if there is one
RESULTS = os.environ.get("REV_RESULTS", "results")
RUN     = False     # False = skip training, only rebuild tables/figures from results/

import os, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, Markdown
from src import report, plots
from scripts.run_jobs import run_stage, stage_tags
warnings.filterwarnings("ignore")
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)
os.makedirs("tables_rev", exist_ok=True)

def export(df, name):
    """Save a table as tables_rev/REV_<name>.csv and show it."""
    path = f"tables_rev/REV_{name}.csv"; df.to_csv(path); display(df); print("saved", path)

In [2]:
df = report.load(RESULTS)
V = report.views(df)
print(len(df), "records")

250 records


In [3]:
from src import metrics as M
rows = []
V = V[V.tag.isin(stage_tags("main"))]
for method, view in [("B3 Dense AE","det_val"), ("M1 Dense+MCD","mc_val"), ("M2 LSTM+MCD","mc_val"),
                     ("M3 Dense+CP","det_cp"), ("M4 LSTM+CP","det_cp")]:
    for ds in ("SKAB","NAB"):
        t = V[(V.method == method) & (V.dataset == ds) & (V.seed == 0)]
        if t.empty: continue
        z = np.load(f"{RESULTS}/arrays/{t.iloc[0].tag}.npz")
        y, f = z["y"], z["file_id"]
        s = z["mc_score"] if view.startswith("mc") else z["det_score"]
        from sklearn.metrics import average_precision_score
        lo, hi = M.file_bootstrap(lambda a, b: average_precision_score(a, b), f, y, s)
        row = {"dataset": ds, "method": method, "AUC-PR": round(average_precision_score(y, s), 3),
               "AUC-PR 95% CI": f"[{lo:.3f}, {hi:.3f}]"}
        if view == "mc_val":
            thr = t.iloc[0]["threshold"]; e = ((s >= thr).astype(int) != y).astype(int)
            lo, hi = M.file_bootstrap(M.auroc, f, e, z["mc_std"])
            row.update({"error-AUROC (std)": round(M.auroc(e, z["mc_std"]), 3),
                        "error-AUROC 95% CI": f"[{lo:.3f}, {hi:.3f}]"})
        if view == "det_cp":
            normal = y == 0
            lo, hi = M.file_bootstrap(lambda p: float(np.mean(p <= 0.05)), f[normal], z["det_cp_p"][normal])
            row.update({"FPR@0.05": round(float(np.mean(z["det_cp_p"][normal] <= 0.05)), 3),
                        "FPR 95% CI": f"[{lo:.3f}, {hi:.3f}]"})
        rows.append(row)
export(pd.DataFrame(rows).set_index(["dataset","method"]), "tableH_bootstrap_ci")

,,AUC-PR,AUC-PR 95% CI,error-AUROC (std),error-AUROC 95% CI,FPR@0.05,FPR 95% CI
dataset,method,,,,,,
SKAB,B3 Dense AE,0.855,"[0.805, 0.899]",NaN,NaN,NaN,NaN
NAB,B3 Dense AE,0.229,"[0.161, 0.331]",NaN,NaN,NaN,NaN
SKAB,M1 Dense+MCD,0.855,"[0.805, 0.898]",0.403,"[0.318, 0.507]",NaN,NaN
NAB,M1 Dense+MCD,0.230,"[0.161, 0.331]",0.720,"[0.642, 0.793]",NaN,NaN
SKAB,M2 LSTM+MCD,0.849,"[0.797, 0.894]",0.432,"[0.347, 0.530]",NaN,NaN
NAB,M2 LSTM+MCD,0.225,"[0.157, 0.326]",0.698,"[0.623, 0.766]",NaN,NaN
SKAB,M3 Dense+CP,0.855,"[0.805, 0.899]",NaN,NaN,0.305,"[0.176, 0.470]"
NAB,M3 Dense+CP,0.229,"[0.161, 0.331]",NaN,NaN,0.059,"[0.028, 0.103]"
SKAB,M4 LSTM+CP,0.849,"[0.799, 0.894]",NaN,NaN,0.336,"[0.208, 0.498]"


saved tables_rev/REV_tableH_bootstrap_ci.csv


### All REV_ outputs produced so far

In [4]:
for d in ("figures_rev", "tables_rev"):
    print(d); [print("  ", f) for f in sorted(os.listdir(d))] if os.path.isdir(d) else print("   (none)")

figures_rev
   REV_M2b_dropout_sweep.png
   REV_M5_dropout_sweep.png
   REV_conformal_false_alarm_rate.png
   REV_fig1_split_composition.png
   REV_label_vs_error_auroc_NAB.png
   REV_label_vs_error_auroc_SKAB.png
   REV_risk_coverage_NAB.png
   REV_risk_coverage_SKAB.png
   REV_window_size_B3.png
   REV_window_size_M1.png
tables_rev
   REV_grid_search_all.csv
   REV_grid_winners.csv
   REV_nab_split_options.csv
   REV_split_old_vs_new.csv
   REV_tableA_detection.csv
   REV_tableB_uncertainty_auroc.csv
   REV_tableC_aurc.csv
   REV_tableD_conformal.csv
   REV_tableE_M2b_sweep.csv
   REV_tableF_M5_sweep.csv
   REV_tableG_window_size.csv
   REV_tableH_bootstrap_ci.csv
